<a href="https://colab.research.google.com/github/seeuni0320/AI-ML/blob/main/Week12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 인공지능과 기계학습 - Week 11: 차원 축소
**Prof. Dong, Suh-Yeon | Div. of Artificial Intelligence Engineering | Sookmyung Women's University**

## 📦 라이브러리 설치 및 임포트

In [ ]:
# 코랩 한글 폰트 설정
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 폰트 캐시 갱신
fm._load_fontmanager(try_read_cache=False)

# 나눔고딕 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

print("한글 폰트 설정 완료!")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

# 한글 폰트 설정 (Colab)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 임포트 완료!")

---
## 8.3 주성분 분석 (PCA)
### 8.3.1–8.3.2 데이터셋 생성 및 SVD로 주성분 구하기

In [ ]:
# 3D 데이터셋 생성
m = 60
X = np.zeros((m, 3))  # 3D 데이터 초기화
np.random.seed(42)
angles = (np.random.rand(m) ** 3 + 0.5) * 2 * np.pi  # 고르지 않은 분포
X[:, 0], X[:, 1] = np.cos(angles), np.sin(angles) * 0.5  # 타원형
X += 0.28 * np.random.randn(m, 3)  # 노이즈 추가
X = Rotation.from_rotvec([np.pi / 29, -np.pi / 20, np.pi / 4]).apply(X)
X += [0.2, 0, 0.2]  # 약간 이동

print("데이터셋 shape:", X.shape)

In [ ]:
# SVD를 사용하여 주성분 구하기
# PCA는 데이터셋의 평균이 0이라고 가정 → 중앙 정렬 필요
X_centered = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(X_centered)

c1 = Vt[0]  # 첫 번째 주성분
c2 = Vt[1]  # 두 번째 주성분

print("첫 번째 주성분 (c1):", c1)
print("두 번째 주성분 (c2):", c2)

### 8.3.3 d 차원으로 투영하기

In [ ]:
# 식 8-2: 훈련 세트를 d차원으로 투영하기  X_d-proj = X @ Wd
# 처음 두 개의 주성분으로 정의된 평면에 훈련 세트 투영
W2 = Vt[:2].T
X2D = X_centered @ W2

print("투영 후 shape:", X2D.shape)

# 시각화
plt.figure(figsize=(6, 4))
plt.scatter(X2D[:, 0], X2D[:, 1], alpha=0.7)
plt.xlabel("$z_1$")
plt.ylabel("$z_2$", rotation=0)
plt.title("SVD를 이용한 PCA 투영 결과 (2D)")
plt.grid(True)
plt.show()

### 8.3.4 사이킷런으로 PCA 사용하기

In [ ]:
from sklearn.decomposition import PCA

# 사이킷런의 PCA 모델은 자동으로 데이터를 중앙에 맞춰줌
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

print("주성분 행렬 (components_):")
print(pca.components_)

### 8.3.5 설명된 분산의 비율

In [ ]:
print("설명된 분산의 비율 (explained_variance_ratio_):")
print(pca.explained_variance_ratio_)
print(f"\n첫 번째 PC: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"두 번째 PC: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"세 번째 PC (나머지): {(1 - pca.explained_variance_ratio_.sum())*100:.1f}%")

### 8.3.6 적절한 차원 수 선택 (MNIST 데이터셋)

In [ ]:
from sklearn.datasets import fetch_openml

# MNIST 데이터셋 로드
mnist = fetch_openml('mnist_784', as_frame=False)
X_train, y_train = mnist.data[:60_000], mnist.target[:60_000]
X_test, y_test = mnist.data[60_000:], mnist.target[60_000:]

print("훈련 세트 shape:", X_train.shape)
print("테스트 세트 shape:", X_test.shape)

In [ ]:
# 차원을 줄이지 않고 PCA 수행 후 95% 분산 보존에 필요한 최소 차원 수 계산
pca = PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1
print(f"분산 95% 보존에 필요한 차원 수: {d}")

In [ ]:
# n_components에 비율(0~1)로 설정하면 더 편리함
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

print("실제 주성분 개수 (n_components_):", pca.n_components_)
print("설명된 분산의 합:", pca.explained_variance_ratio_.sum())

In [ ]:
# 설명된 분산을 차원 수에 대한 함수로 그리기 (그림 8-8)
pca_full = PCA()
pca_full.fit(X_train)
cumsum = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(6, 4))
plt.plot(cumsum, linewidth=3)
plt.axis([0, 400, 0, 1])
plt.xlabel("Dimension")
plt.ylabel("Explained Variance")
plt.plot([d, d], [0, 0.95], "k:")
plt.plot([0, d], [0.95, 0.95], "k:")
plt.plot(d, 0.95, "ko")
plt.annotate("Elbow", xy=(65, 0.85), xytext=(70, 0.7),
             arrowprops=dict(arrowstyle="->"))
plt.grid(True)
plt.title("차원 수에 대한 함수로 나타낸 설명된 분산")
plt.show()

### 8.3.6 PCA + RandomForest 하이퍼파라미터 탐색

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

clf = make_pipeline(PCA(random_state=42),
                    RandomForestClassifier(random_state=42))

param_distrib = {
    "pca__n_components": np.arange(10, 80),
    "randomforestclassifier__n_estimators": np.arange(50, 500)
}

rnd_search = RandomizedSearchCV(clf, param_distrib, n_iter=10, cv=3,
                                random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])

print("최적 파라미터:", rnd_search.best_params_)

### 8.3.7 압축을 위한 PCA (역변환)

In [ ]:
# 식 8-3: 원본의 차원 수로 되돌리는 PCA 역변환  X_recovered = X_d-proj @ Wd^T
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)
X_recovered = pca.inverse_transform(X_reduced)

print(f"원본 shape: {X_train.shape}")
print(f"압축 후 shape: {X_reduced.shape}")
print(f"복원 후 shape: {X_recovered.shape}")

In [ ]:
# 원본과 압축 복원 이미지 비교 (그림 8-9)
plt.figure(figsize=(7, 4))
for idx, X_vis in enumerate((X_train[::2100], X_recovered[::2100])):
    plt.subplot(1, 2, idx + 1)
    plt.title(["원본", "압축 후 복원"][idx])
    for row in range(5):
        for col in range(5):
            plt.imshow(X_vis[row * 5 + col].reshape(28, 28),
                       cmap="binary", vmin=0, vmax=255,
                       extent=(row, row + 1, col, col + 1))
    plt.axis([0, 5, 0, 5])
    plt.axis("off")
plt.suptitle("분산의 95%가 유지된 MNIST 압축 (그림 8-9)")
plt.tight_layout()
plt.show()

### 8.3.8 랜덤 PCA

In [ ]:
# svd_solver='randomized': 근사값을 빠르게 계산
rnd_pca = PCA(n_components=154, svd_solver="randomized", random_state=42)
X_reduced = rnd_pca.fit_transform(X_train)
print("랜덤 PCA 결과 shape:", X_reduced.shape)

### 8.3.9 점진적 PCA (Incremental PCA)

In [ ]:
from sklearn.decomposition import IncrementalPCA

# 훈련 세트를 100개의 미니배치로 나누어 처리
n_batches = 100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)  # partial_fit()을 미니배치마다 호출

X_reduced = inc_pca.transform(X_train)
print("점진적 PCA 결과 shape:", X_reduced.shape)

In [ ]:
import os

# memmap을 사용한 점진적 PCA (대용량 데이터 처리)
filename = "my_mnist.mmap"
X_mmap = np.memmap(filename, dtype='float32', mode='write', shape=X_train.shape)
X_mmap[:] = X_train
X_mmap.flush()

# 읽기 모드로 열어서 IncrementalPCA 적용
X_mmap = np.memmap(filename, dtype="float32", mode="readonly").reshape(-1, 784)
batch_size = X_mmap.shape[0] // n_batches
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
inc_pca.fit(X_mmap)

print("memmap 기반 점진적 PCA 완료!")
os.remove(filename)  # 임시 파일 삭제

---
## 8.4 랜덤 투영

In [ ]:
from sklearn.random_projection import johnson_lindenstrauss_min_dim

# 존슨-린덴스트라우스 방정식으로 필요한 최소 차원 수 계산
m, epsilon = 5_000, 0.1
d = johnson_lindenstrauss_min_dim(m, eps=epsilon)
print(f"샘플 수={m}, ε={epsilon} → 최소 차원 수 d = {d}")

In [ ]:
# 수동 랜덤 투영
n = 20_000
np.random.seed(42)
P = np.random.randn(d, n) / np.sqrt(d)  # 표준 편차 = 분산의 제곱근

X_fake = np.random.randn(m, n)  # 가짜 데이터셋
X_reduced = X_fake @ P.T
print(f"원본 shape: {X_fake.shape} → 투영 후 shape: {X_reduced.shape}")

In [ ]:
from sklearn.random_projection import GaussianRandomProjection
import scipy

# 사이킷런 GaussianRandomProjection
gaussian_rnd_proj = GaussianRandomProjection(eps=epsilon, random_state=42)
X_reduced_2 = gaussian_rnd_proj.fit_transform(X_fake)

# 수동 결과와 동일한지 확인
are_close = np.allclose(X_reduced, X_reduced_2, atol=1e-7)
print("수동 결과와 동일한가?", are_close)

# 역변환 (유사역행렬 사용)
components_pinv = np.linalg.pinv(gaussian_rnd_proj.components_)
X_recovered = X_reduced_2 @ components_pinv.T
print("역변환 후 shape:", X_recovered.shape)

수동 결과와 동일한가? True


In [ ]:
from sklearn.random_projection import SparseRandomProjection

# SparseRandomProjection: 희소 행렬 → 메모리 효율적
sparse_rnd_proj = SparseRandomProjection(eps=epsilon, random_state=42)
X_reduced_sparse = sparse_rnd_proj.fit_transform(X_fake)
print("SparseRandomProjection 결과 shape:", X_reduced_sparse.shape)

---
## 8.5 지역 선형 임베딩 (LLE)

In [ ]:
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import LocallyLinearEmbedding

# 스위스 롤 데이터셋 생성
X_swiss, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)

# LLE로 펼치기
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_unrolled = lle.fit_transform(X_swiss)

print("스위스 롤 shape:", X_swiss.shape)
print("LLE 펼친 후 shape:", X_unrolled.shape)

In [ ]:
from matplotlib.colors import ListedColormap

darker_hot = ListedColormap(plt.cm.hot(np.linspace(0, 0.8, 256)))

fig = plt.figure(figsize=(12, 5))

# 원본 스위스 롤 (3D)
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2],
            c=t, cmap=darker_hot)
ax1.view_init(10, -70)
ax1.set_title("스위스 롤 (3D)")

# LLE 펼친 결과 (2D) - 그림 8-10
ax2 = fig.add_subplot(122)
ax2.scatter(X_unrolled[:, 0], X_unrolled[:, 1], c=t, cmap=darker_hot)
ax2.set_xlabel("$z_1$")
ax2.set_ylabel("$z_2$", rotation=0)
ax2.set_title("LLE로 펼쳐진 스위스 롤 (그림 8-10)")
ax2.grid(True)

plt.tight_layout()
plt.show()

---
## 8.6 다른 차원 축소 기법 비교 (MDS, Isomap, t-SNE)

In [ ]:
from sklearn.manifold import MDS, TSNE, Isomap

# 각 방법으로 스위스 롤 2D 축소
methods = [
    ("MDS", MDS(n_components=2, random_state=42)),
    ("Isomap", Isomap(n_components=2)),
    ("t-SNE", TSNE(n_components=2, init="random", learning_rate="auto", random_state=42)),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, method) in zip(axes, methods):
    X_2d = method.fit_transform(X_swiss)
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=t, cmap=darker_hot, alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel("$z_1$")
    ax.set_ylabel("$z_2$", rotation=0)
    ax.grid(True)

plt.suptitle("여러 기법으로 스위스 롤을 2D로 축소 (그림 8-11)")
plt.tight_layout()
plt.show()

---
## 연습문제 9: MNIST에 PCA 적용하기

In [ ]:
# a. MNIST 데이터셋 로드 (위에서 이미 로드됨)
print("훈련 세트:", X_train.shape, "/ 테스트 세트:", X_test.shape)

In [ ]:
import time
from sklearn.metrics import accuracy_score

# b. 랜덤 포레스트 분류기 (PCA 없이)
rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)

t0 = time.time()
rnd_clf.fit(X_train, y_train)
t1 = time.time()
print(f"훈련 시간 (PCA 없이): {t1 - t0:.1f}초")

y_pred = rnd_clf.predict(X_test)
print(f"정확도 (PCA 없이): {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# c. PCA로 차원 축소 후 랜덤 포레스트 훈련
pca = PCA(n_components=0.95)
X_train_reduced = pca.fit_transform(X_train)
print(f"PCA 후 차원: {X_train_reduced.shape[1]}개")

rnd_clf_with_pca = RandomForestClassifier(n_estimators=100, random_state=42)

t0 = time.time()
rnd_clf_with_pca.fit(X_train_reduced, y_train)
t1 = time.time()
print(f"훈련 시간 (PCA 있음): {t1 - t0:.1f}초")

In [ ]:
# d. 테스트 세트 평가
X_test_reduced = pca.transform(X_test)
y_pred = rnd_clf_with_pca.predict(X_test_reduced)
print(f"정확도 (PCA 있음): {accuracy_score(y_test, y_pred):.4f}")
print("\n→ PCA를 적용하면 훈련 속도는 달라지지만 정확도가 약간 감소할 수 있음")

In [ ]:
from sklearn.linear_model import SGDClassifier

# e. SGDClassifier로 비교
# PCA 없이
sgd_clf = SGDClassifier(random_state=42)
t0 = time.time()
sgd_clf.fit(X_train, y_train)
t1 = time.time()
print(f"SGD 훈련 시간 (PCA 없이): {t1 - t0:.1f}초")
y_pred = sgd_clf.predict(X_test)
print(f"SGD 정확도 (PCA 없이): {accuracy_score(y_test, y_pred):.4f}")

# PCA 있음
sgd_clf_pca = SGDClassifier(random_state=42)
t0 = time.time()
sgd_clf_pca.fit(X_train_reduced, y_train)
t1 = time.time()
print(f"\nSGD 훈련 시간 (PCA 있음): {t1 - t0:.1f}초")
y_pred = sgd_clf_pca.predict(X_test_reduced)
print(f"SGD 정확도 (PCA 있음): {accuracy_score(y_test, y_pred):.4f}")

---
## 연습문제 10: MNIST에 t-SNE 적용하기

In [ ]:
# a. t-SNE로 MNIST 2D 축소 (샘플 5000개 사용)
X_sample, y_sample = X_train[:5000], y_train[:5000]

tsne = TSNE(n_components=2, init="random", learning_rate="auto", random_state=42)
X_reduced_tsne = tsne.fit_transform(X_sample)

# 색상 산점도
plt.figure(figsize=(13, 10))
plt.scatter(X_reduced_tsne[:, 0], X_reduced_tsne[:, 1],
            c=y_sample.astype(np.int8), cmap="jet", alpha=0.5)
plt.axis('off')
plt.colorbar()
plt.title("t-SNE: MNIST 2D 시각화")
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from matplotlib.offsetbox import AnnotationBbox, OffsetImage

def plot_digits(X, y, min_distance=0.04, images=None, figsize=(13, 10)):
    X_normalized = MinMaxScaler().fit_transform(X)
    neighbors = np.array([[10., 10.]])
    plt.figure(figsize=figsize)
    cmap = plt.cm.jet
    digits = np.unique(y)
    for digit in digits:
        plt.scatter(X_normalized[y == digit, 0], X_normalized[y == digit, 1],
                    c=[cmap(float(digit) / 9)], alpha=0.5)
    plt.axis("off")
    ax = plt.gca()
    for index, image_coord in enumerate(X_normalized):
        closest_distance = np.linalg.norm(neighbors - image_coord, axis=1).min()
        if closest_distance > min_distance:
            neighbors = np.r_[neighbors, [image_coord]]
            if images is None:
                plt.text(image_coord[0], image_coord[1], str(int(y[index])),
                         color=cmap(float(y[index]) / 9),
                         fontdict={"weight": "bold", "size": 16})
            else:
                image = images[index].reshape(28, 28)
                imagebox = AnnotationBbox(OffsetImage(image, cmap="binary"),
                                         image_coord)
                ax.add_artist(imagebox)

In [ ]:
# b-1: 숫자 레이블로 표시
plot_digits(X_reduced_tsne, y_sample.astype(int))
plt.title("t-SNE: 숫자 레이블로 표시")
plt.show()

In [ ]:
# b-2: 실제 이미지로 표시 (그림 8-11 스타일)
plot_digits(X_reduced_tsne, y_sample.astype(int),
            images=X_sample, figsize=(35, 25))
plt.title("t-SNE: 실제 이미지로 표시", fontsize=20)
plt.show()